In [8]:
import fastf1
from fastf1 import get_event_schedule, get_session
import pandas as pd
import os
import logging
import re

In [9]:
# ========== CONFIG ==========
SEASON = 2018
WET_LAP_THRESHOLD = 0.1
cache_dir = '../.fastf1_cache'
output_dir = '../data/processed'
os.makedirs(cache_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

In [10]:
fastf1.Cache.enable_cache(cache_dir)

compound_map = {
    'SOFT': 'Soft', 'MEDIUM': 'Medium', 'HARD': 'Hard',
    'INTERMEDIATE': 'Intermediate', 'WET': 'Wet'
}

def slugify(text):
    """Basic slugify for safe filenames."""
    return re.sub(r'[^\w]+', '_', text.lower()).strip('_')


In [11]:
# ========== LOGGING ==========
logging.basicConfig(filename=f'fastf1_data_export_{SEASON}.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [12]:
# ========== DATA START ==========
schedule = get_event_schedule(SEASON, include_testing=False)
race_events = schedule.copy()
combined_export_path = f"{output_dir}/all_races_combined_{SEASON}.csv"

all_races = []

In [13]:
for _, row in race_events.iterrows():
    round_num = row['RoundNumber']
    event_name = row['EventName']
    gp_name = slugify(event_name)
    event_date = row['EventDate']

    try:
        session = get_session(SEASON, round_num, 'R')
        session.load()
    except Exception as e:
        logging.error(f"[LOAD FAIL] {gp_name}: {e}")
        print(f"❌ Failed to load session for {gp_name} — {e}")
        continue

    try:
        laps = session.laps.reset_index(drop=True)
        selected_cols = [
            'Driver', 'Team', 'LapNumber', 'LapTime',
            'Sector1Time', 'Sector2Time', 'Sector3Time',
            'Compound', 'TyreLife', 'Stint',
            'PitInTime', 'PitOutTime', 'TrackStatus',
            'IsAccurate', 'Time'
        ]
        lap_data = laps[selected_cols].copy()

        for col in ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']:
            lap_data[f'{col}Seconds'] = lap_data[col].dt.total_seconds()
        lap_data['LapStartTime'] = lap_data['Time'].dt.total_seconds()

        # Weather Merge (Safe)
        try:
            weather = session.weather_data.rename(columns={'Time': 'WeatherTime'})
            weather['WeatherTime'] = weather['WeatherTime'].dt.total_seconds()
            lap_data = pd.merge_asof(
                lap_data.sort_values('LapStartTime'),
                weather.sort_values('WeatherTime'),
                left_on='LapStartTime', right_on='WeatherTime',
                direction='nearest'
            )
        except Exception:
            logging.warning(f"No weather data for {gp_name}")
            lap_data['Rainfall'] = 0
            lap_data['AirTemp'] = None
            lap_data['Humidity'] = None
            lap_data['WindSpeed'] = None

        # Rain flags
        lap_data['IsWetLap'] = lap_data['Rainfall'] > WET_LAP_THRESHOLD
        lap_data['IsDryLap'] = lap_data['Rainfall'] <= WET_LAP_THRESHOLD
        lap_data['IsWetRace'] = lap_data['IsWetLap'].any()

        # Circuit Metadata
        try:
            circuit_info = session.get_circuit_info()
            lap_data['CircuitName'] = circuit_info.name
            lap_data['CircuitShort'] = circuit_info.location
            lap_data['CircuitCountry'] = circuit_info.country
            lap_data['TrackLengthKM'] = circuit_info.length / 1000 if circuit_info.length else None
            lap_data['AltitudeM'] = circuit_info.altitude
        except Exception:
            event = session.event
            lap_data['CircuitName'] = event.get('OfficialEventName', event_name)
            lap_data['CircuitShort'] = event.get('Location', 'Unknown')
            lap_data['CircuitCountry'] = event.get('Country', 'Unknown')
            lap_data['TrackLengthKM'] = None
            lap_data['AltitudeM'] = None

        lap_data['CircuitType'] = lap_data['CircuitShort'].apply(lambda name: (
            "Street" if isinstance(name, str) and any(x in name.lower() for x in ['monaco', 'baku', 'miami', 'jeddah'])
            else "Hybrid" if isinstance(name, str) and 'marina' in name.lower()
            else "Permanent"
        ))

        # Type & Cleanup
        lap_data['TrackStatus'] = lap_data['TrackStatus'].astype(str)
        lap_data['IsAccurate'] = lap_data['IsAccurate'].astype(bool)
        lap_data['LapNumber'] = lap_data['LapNumber'].fillna(0).astype(int)
        lap_data['TyreLife'] = lap_data['TyreLife'].fillna(0).astype(int)
        lap_data['Stint'] = lap_data['Stint'].fillna(0).astype(int)

        lap_data.dropna(subset=[
            'LapTimeSeconds', 'Sector1TimeSeconds', 'Sector2TimeSeconds',
            'Sector3TimeSeconds', 'Compound'
        ], inplace=True)

        lap_data['Compound'] = lap_data['Compound'].str.upper().map(compound_map).fillna(lap_data['Compound'])

        # Pit Info
        lap_data['PitLap'] = lap_data.apply(
            lambda row: row['LapNumber'] if pd.notna(row['PitInTime']) else None, axis=1
        )
        lap_data['PitDuration'] = (lap_data['PitOutTime'] - lap_data['PitInTime']).dt.total_seconds()

        # Sector Features
        lap_data['Sector1Pct'] = lap_data['Sector1TimeSeconds'] / lap_data['LapTimeSeconds']
        lap_data['Sector2Pct'] = lap_data['Sector2TimeSeconds'] / lap_data['LapTimeSeconds']
        lap_data['Sector3Pct'] = lap_data['Sector3TimeSeconds'] / lap_data['LapTimeSeconds']
        lap_data['BestSector'] = lap_data[['Sector1TimeSeconds', 'Sector2TimeSeconds', 'Sector3TimeSeconds']].idxmin(axis=1)
        lap_data['BestSector'] = lap_data['BestSector'].str.extract(r'(\d)').astype(int)
        lap_data['IsValidLap'] = (lap_data['TrackStatus'] == 'Green') & lap_data['IsAccurate']

        # Metadata
        lap_data['GrandPrix'] = event_name
        lap_data['GP_Slug'] = gp_name
        lap_data['SeasonYear'] = SEASON
        lap_data['EventDate'] = pd.to_datetime(event_date)

        if 'CarNumber' in laps.columns:
            lap_data['CarNumber'] = laps['CarNumber']

        # Stint Summary
        stint_summary = lap_data.groupby(['Driver', 'Stint']).agg(
            AvgLapTime=('LapTimeSeconds', 'mean'),
            StintLength=('LapNumber', 'count')
        ).reset_index()
        lap_data = lap_data.merge(stint_summary, on=['Driver', 'Stint'], how='left')

        stint_max_map = lap_data.groupby('Driver')['Stint'].max().to_dict()
        lap_data['StintType'] = lap_data.apply(
            lambda row: "Opening" if row['Stint'] == 1 else
                        "Closing" if row['Stint'] == stint_max_map.get(row['Driver'], 3) else "Mid", axis=1
        )

        # Delta to Driver’s Fastest Lap
        fastest_per_driver = lap_data.groupby("Driver")["LapTimeSeconds"].min().to_dict()
        lap_data['DeltaToFastestLap'] = lap_data.apply(
            lambda row: row["LapTimeSeconds"] - fastest_per_driver.get(row["Driver"], row["LapTimeSeconds"]),
            axis=1
        )

        # Flags
        lap_data["IsSC"] = lap_data["TrackStatus"].str.contains("4").fillna(False)
        lap_data["IsVSC"] = lap_data["TrackStatus"].str.contains("8").fillna(False)
        lap_data["IsRedFlag"] = lap_data["TrackStatus"].str.contains("16").fillna(False)
        race_max_lap = lap_data["LapNumber"].max()
        lap_data["IsDNF"] = lap_data["LapNumber"] < (race_max_lap - 3)

        # Export CSV
        race_csv = f"{output_dir}/race_summary_{SEASON}_{gp_name}.csv"
        lap_data.to_csv(race_csv, index=False)
        all_races.append(lap_data)
        logging.info(f"[EXPORT OK] {gp_name}")
        print(f"✅ Exported: {gp_name}")

    except Exception as e:
        logging.error(f"[PROCESS FAIL] {gp_name}: {e}")
        print(f"❌ Failed processing for {gp_name} — {e}")


core           INFO 	Loading data for Australian Grand Prix - Race [v3.5.3]


req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
core        WARNING 	Car position data is unavailable!
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 5 completed the race distance 00:00.123000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3

✅ Exported: australian_grand_prix


core        WARNING 	Driver 27: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver  2: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver  9: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver 31: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver 55: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver 16: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver  8: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver 18: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver 35: Lap timing integrity check failed for 1 lap(s)
core        WARNING 	Driver 28: Lap timing integrity check failed for 1 lap(s)
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
core        WARNING 	Car telemetry data is unavailable!
req            INFO 	No cached data foun

✅ Exported: bahrain_grand_prix


...

✅ Exported: mexican_grand_prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 44 completed the race distance 00:00.157000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '7', '3', '77', '5', '16', '8', '20', '11', '28', '55', '10', '2', '31', '35', '14', '18', '27', '9']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
req  

✅ Exported: brazilian_grand_prix


core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 44 completed the race distance 00:00.074000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['44', '5', '33', '3', '77', '55', '16', '11', '8', '20', '14', '28', '18', '2', '35', '10', '31', '9', '7', '27']


✅ Exported: abu_dhabi_grand_prix


In [14]:
# Combined Export
if all_races:
    combined_df = pd.concat(all_races, ignore_index=True)
    combined_df.to_csv(combined_export_path, index=False)
    print(f"\n🎉 Combined season export → {combined_export_path}")
else:
    print("⚠️ No races were processed.")


🎉 Combined season export → ../data/processed/all_races_combined_2018.csv
